# 03 — Token embeddings and sinusoidal positions

> **Status:** complete.

- **Mapped issue:** [#4](https://github.com/majorgilles/transformer-2017-reproduction/issues/4)
- **Depends on:** `02_shared_bpe.ipynb` / issue #3.
- **Paper reference:** section 3.4, *Embeddings and Softmax*.


In [1]:
#| default_exp embeddings


## Goal

Convert token IDs to scaled vectors and add paper-faithful sinusoidal positions.


## Token IDs become vectors

A token ID selects one learned row from an embedding table. The table has one row
per vocabulary token and `d_model` columns per row.

In [2]:
import torch
from torch import nn

torch.manual_seed(0)

token_ids = torch.tensor([[0, 2, 1]])
embedding = nn.Embedding(num_embeddings=4, embedding_dim=6)
token_vectors = embedding(token_ids)

print(f"token ID shape: {tuple(token_ids.shape)}")
print(f"embedding table shape: {tuple(embedding.weight.shape)}")
print(f"token vector shape: {tuple(token_vectors.shape)}")
print(token_vectors)

token ID shape: (1, 3)
embedding table shape: (4, 6)
token vector shape: (1, 3, 6)
tensor([[[-1.1258, -1.1524, -0.2506, -0.4339,  0.8487,  0.6920],
         [ 0.1665,  0.8744, -0.1435, -0.1116,  0.9318,  1.2590],
         [-0.3160, -2.1152,  0.4681, -0.1577,  1.4437,  0.2660]]],
       grad_fn=<EmbeddingBackward0>)


## Why scale token embeddings?

The Transformer combines token identity and position by adding their vectors
element by element. Sinusoidal position values are fixed between approximately
`-1` and `1`, while token embeddings are learned.

Following section 3.4 of *Attention Is All You Need*, we multiply each token
embedding by:

$$
\sqrt{d_{\text{model}}}
$$

This width-aware scale gives the token signal a useful magnitude before the
position vector is added. It does not change the token, vector shape, or
direction; it only changes the vector's magnitude.

In [3]:
#| export
import torch
from torch import nn


class TokenEmbedding(nn.Module):
    """Look up token vectors and apply the paper's embedding scale.

    Each token ID selects one learned vector with `d_model` values. Following
    section 3.4 of the Transformer paper, the vector is multiplied by
    `sqrt(d_model)` before positional information is added.
    """

    def __init__(self, vocab_size: int, d_model: int) -> None:
        super().__init__()
        # One learned row per vocabulary token; each row has `d_model` values.
        self.embedding = nn.Embedding(vocab_size, d_model)
        # Paper section 3.4: strengthen token identity relative to the fixed
        # sinusoidal values that will be added to this vector.
        self.scale = d_model**0.5

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Return scaled vectors with shape (batch, sequence, d_model)."""
        # Scaling changes magnitude but preserves shape and token order.
        return self.embedding(token_ids) * self.scale

In [4]:
torch.manual_seed(0)

scaled_embedding = TokenEmbedding(vocab_size=4, d_model=6)
raw_vectors = scaled_embedding.embedding(token_ids)
scaled_vectors = scaled_embedding(token_ids)

expected_vectors = raw_vectors * scaled_embedding.scale

assert scaled_vectors.shape == (1, 3, 6)
assert torch.allclose(scaled_vectors, expected_vectors)

raw_norm = raw_vectors[0, 0].norm().item()
scaled_norm = scaled_vectors[0, 0].norm().item()

print("Embedding scaling demonstration:")
print(f"  scale: {scaled_embedding.scale:.3f}")
print(f"  shape before/after scaling: {tuple(scaled_vectors.shape)}")
print(f"  first-token norm before: {raw_norm:.3f}")
print(f"  first-token norm after: {scaled_norm:.3f}")

Embedding scaling demonstration:
  scale: 2.449
  shape before/after scaling: (1, 3, 6)
  first-token norm before: 2.011
  first-token norm after: 4.927


## Embeddings alone do not represent order

An embedding lookup depends only on the token ID. If the same token appears at
three positions, all three positions receive the same vector. The Transformer
therefore needs a separate position signal before attention can distinguish
where each occurrence appears.

In [5]:
repeated_token_ids = torch.tensor([[2, 2, 2]])
repeated_vectors = scaled_embedding(repeated_token_ids)

assert torch.allclose(repeated_vectors[0, 0], repeated_vectors[0, 1])
assert torch.allclose(repeated_vectors[0, 1], repeated_vectors[0, 2])

print("Repeated-token evidence:")
print(f"  token IDs: {repeated_token_ids.tolist()}")
print(f"  first three values at position 0: {repeated_vectors[0, 0, :3]}")
print(f"  first three values at position 1: {repeated_vectors[0, 1, :3]}")
print(f"  first three values at position 2: {repeated_vectors[0, 2, :3]}")

Repeated-token evidence:
  token IDs: [[2, 2, 2]]
  first three values at position 0: tensor([ 0.4077,  2.1418, -0.3514], grad_fn=<SliceBackward0>)
  first three values at position 1: tensor([ 0.4077,  2.1418, -0.3514], grad_fn=<SliceBackward0>)
  first three values at position 2: tensor([ 0.4077,  2.1418, -0.3514], grad_fn=<SliceBackward0>)


## Sinusoidal positions create an order signal

For position `pos` and dimension pair `i`, the paper defines:

$$
PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

$$
PE(pos, 2i + 1) = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

Each sine/cosine pair changes at a different speed. Early dimensions change
quickly across nearby positions, while later dimensions change slowly. Together,
these patterns give every position a distinct vector with `d_model` values.

### Reading the table slices

A slice uses `start:stop:step`. When `stop` is omitted, Python continues to
the end of that axis. In `position_table[:, 0::2]`, the first `:` selects every
position row, while `0::2` selects even columns `0, 2, 4, ...`. Those columns
receive sine values. Similarly, `position_table[:, 1::2]` selects odd columns
`1, 3, 5, ...`, which receive cosine values.

For `d_model = 6`, the columns form three sine/cosine pairs: `(0, 1)`, `(2, 3)`,
and `(4, 5)`.

In [6]:
max_length = 4
d_model = 6

# `pos` in the paper: positions 0, 1, 2, 3.
positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
print(positions.shape)

print(d_model // 2)
# `i` in the paper: one index per sine/cosine pair.
pair_indices = torch.arange(d_model // 2, dtype=torch.float32)

# The exact denominator from 10000^(2i / d_model).
denominators = 10000.0 ** ((2 * pair_indices) / d_model)

# The angle inside both sin(...) and cos(...).
angles = positions / denominators

position_table = torch.zeros(max_length, d_model)

# PE(pos, 2i): keep every position row (`:`) and fill even columns
# 0, 2, 4, ... (`0::2`) with sine values.
position_table[:, 0::2] = torch.sin(angles)

# PE(pos, 2i + 1): keep every row and fill odd columns
# 1, 3, 5, ... (`1::2`) with cosine values.
position_table[:, 1::2] = torch.cos(angles)

assert position_table.shape == (4, 6)
assert torch.allclose(
    position_table[0],
    torch.tensor([0.0, 1.0, 0.0, 1.0, 0.0, 1.0]),
)

print("Sinusoidal position table:")
print(position_table)

torch.Size([4, 1])
3
Sinusoidal position table:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000]])


In [7]:
#| export
def sinusoidal_position_table(max_length: int, d_model: int) -> torch.Tensor:
    """Build the paper's fixed sine/cosine position vectors.

    The returned tensor has shape `(max_length, d_model)`. Each row represents
    one sequence position. Even columns contain sine values and their adjacent
    odd columns contain cosine values computed with the same frequency.
    """

    if d_model % 2 != 0:
        raise ValueError("d_model must be even so sine/cosine dimensions can pair")

    positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
    pair_indices = torch.arange(d_model // 2, dtype=torch.float32)

    # This directly represents the paper's denominator: 10000^(2i / d_model).
    denominators = 10000.0 ** ((2 * pair_indices) / d_model)
    angles = positions / denominators

    table = torch.zeros(max_length, d_model)

    # PE(pos, 2i): `:` keeps every position row; `0::2` selects even
    # dimensions 0, 2, 4, ... for sine values.
    table[:, 0::2] = torch.sin(angles)

    # PE(pos, 2i + 1): `1::2` selects the adjacent odd dimensions
    # 1, 3, 5, ... for cosine values at the same frequencies.
    table[:, 1::2] = torch.cos(angles)

    return table

In [8]:
reusable_position_table = sinusoidal_position_table(
    max_length=4,
    d_model=6,
)

assert torch.allclose(reusable_position_table, position_table)

print("Reusable position table matches the paper-formula experiment:")
print(reusable_position_table)

Reusable position table matches the paper-formula experiment:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000]])


## Add fixed positions to token vectors

The position table is fixed rather than learned. We register it as a PyTorch
buffer so it moves with the model and is saved with the model, but receives no
optimizer updates.

For an input with shape `(batch, sequence, d_model)`, we select the first
`sequence` rows from the table and add a batch axis. The resulting shape
`(1, sequence, d_model)` broadcasts across every item in the batch.

In [9]:
#| export
class SinusoidalPositionalEncoding(nn.Module):
    """Add the paper's fixed sinusoidal position vectors to token vectors.

    The fixed table is stored as a buffer rather than a trainable parameter.
    During `forward`, the module selects one table row per token position and
    broadcasts those rows across every sequence in the batch.
    """

    position_table: torch.Tensor

    def __init__(self, max_length: int, d_model: int) -> None:
        super().__init__()

        table = sinusoidal_position_table(
            max_length=max_length,
            d_model=d_model,
        )

        # A buffer is saved and moved with the module but is not learned.
        self.register_buffer("position_table", table)

    def forward(self, token_vectors: torch.Tensor) -> torch.Tensor:
        """Add positions to a `(batch, sequence, d_model)` tensor."""

        sequence_length = token_vectors.shape[1]

        if sequence_length > self.position_table.shape[0]:
            raise ValueError("sequence length exceeds the position table")

        # Select one position row per token, then add a batch axis so the same
        # position pattern broadcasts across every sequence in the batch.
        positions = self.position_table[:sequence_length].unsqueeze(0)

        return token_vectors + positions

### Proving what the module adds

The module computes `combined = tokens + positions`. Subtracting the original
token vectors therefore recovers the position vectors: `combined - tokens =
positions`. Comparing those recovered offsets with the position table proves
that the module selects the correct rows, broadcasts them across the batch, and
does not otherwise alter the token vectors.

Because these calculations use 32-bit floating-point numbers, addition followed
by subtraction can leave tiny rounding differences. `torch.allclose` uses an
absolute tolerance of `1e-6` here: differences smaller than one millionth are
accepted, while a wrong position or formula still fails.

In [10]:
positional_encoding = SinusoidalPositionalEncoding(
    max_length=4,
    d_model=6,
)

# Repeat the same three-token sequence to create a batch of two.
token_batch = scaled_vectors.repeat(2, 1, 1)
combined_vectors = positional_encoding(token_batch)

# Subtraction reveals exactly which position vectors were added.
position_offsets = combined_vectors - token_batch

# Select the three expected table rows, add a batch axis, and explicitly
# expand them to the same `(2, 3, 6)` shape as the recovered offsets.
expected_offsets = reusable_position_table[:3].unsqueeze(0).expand_as(position_offsets)

assert combined_vectors.shape == (2, 3, 6)

# Floating-point addition and subtraction can leave differences smaller than
# one millionth. The tolerance ignores that noise, not real formula errors.
assert torch.allclose(
    position_offsets,
    expected_offsets,
    rtol=0.0,
    atol=1e-6,
)

# The table belongs to the module as a fixed buffer, with no learned parameters.
assert "position_table" in dict(positional_encoding.named_buffers())
assert list(positional_encoding.parameters()) == []

print("Token + position demonstration:")
print(f"  token batch shape: {tuple(token_batch.shape)}")
print(f"  combined shape: {tuple(combined_vectors.shape)}")
print("  first three offset values at each position:")
for position in range(3):
    print(f"    position {position}: {position_offsets[0, position, :3]}")

Token + position demonstration:
  token batch shape: (2, 3, 6)
  combined shape: (2, 3, 6)
  first three offset values at each position:
    position 0: tensor([0., 1., 0.], grad_fn=<SliceBackward0>)
    position 1: tensor([0.8415, 0.5403, 0.0464], grad_fn=<SliceBackward0>)
    position 2: tensor([ 0.9093, -0.4161,  0.0927], grad_fn=<SliceBackward0>)


## What this notebook demonstrates

- Token IDs select learned vectors with `d_model` values.
- Token embeddings are multiplied by $\sqrt{d_{\text{model}}}$.
- Fixed sine/cosine vectors give each sequence position an order signal.
- Token and position vectors combine without changing tensor shape.


## Focused evidence

- Embedding output has shape `(batch, sequence, d_model)`.
- Scaling changes vector magnitude but preserves shape and token order.
- Repeated token IDs receive identical vectors before positions are added.
- Position zero alternates between `sin(0) = 0` and `cos(0) = 1`.
- The positional table is a fixed buffer with no learned parameters.
- Subtracting token vectors from combined vectors recovers the expected positions.


## Explicitly deferred

Masks, attention scores, multi-head projections, encoder, and decoder.


## Review checkpoint

Review the paper equations, embedding scale, tensor shapes, positional values,
and focused assertions before closing issue #4.
